In [ ]:
import polars as pl
from datetime import date, timedelta
from sklearn.metrics import classification_report

from fleetsense.features.data_loader import get_dataset, FEATURES, TARGET_COLUMN
from fleetsense.model.base_model import (
    load_baseline_model,
)
from fleetsense.config import DATA_DATASET

from fleetsense.monitoring.distribution_monitoring import (
    load_baselines,
)
import matplotlib.pyplot as plt
import sys
from fleetsense.monitoring.distribution_monitoring import (
    monitor_all_features,
    add_weighted_psi,
    check_drift,
    weighted_drift_score,
)

sys.path.append("..")
from datetime import datetime
from scripts.train_model import train, load_permutation_importance

In [ ]:
FULL_DATA_PATH = DATA_DATASET / "vessel_weekly_features.csv"

df = get_dataset(FULL_DATA_PATH)  # ensure the dataset is loaded

print(df.shape)

In [ ]:
SAMPLE_DATA_PATH = DATA_DATASET / "vessel_weekly_features_sample.csv"
TRAIN_START = date(2025, 6, 1)  # your known baseline window, same as production training
TRAIN_END = date(2025, 8, 31)
train(start=TRAIN_START, end=TRAIN_END, data_path=FULL_DATA_PATH)  # actually calls training pipeline

model = load_baseline_model()  # the actual saved artifact
baselines = load_baselines()  # the actual saved baseline
importance_df = load_permutation_importance()
weights = importance_df["drift_weight"].to_dict()

In [ ]:
df = get_dataset(FULL_DATA_PATH)  # load the dataset for evaluation
df = df.with_columns(pl.col("timestamp").str.to_datetime("%Y-%m-%dT%H:%M:%S%.f"))

reports = {}
for week_start in sorted(df["timestamp"].unique().to_list()):
    week_end = week_start + timedelta(days=7)
    test_df = df.filter(pl.col("timestamp").is_between(week_start, week_end, closed="left"))
    if test_df.is_empty():
        continue

    X_test = test_df[FEATURES]
    y_test = test_df[TARGET_COLUMN]

    results = model.predict(X_test)

    report = classification_report(y_test, results, output_dict=True)
    reports[week_start.strftime("%Y-%m-%d")] = report

In [ ]:
%matplotlib inline

classes = ["Cargo", "Tanker", "Fishing", "Passenger", "Tug"]
weeks_sorted = sorted(w for w in reports if w > "2025-09-01")

fig, ax = plt.subplots(figsize=(14, 6))
for cls in classes:
    f1_scores = [reports[w].get(cls, {}).get("f1-score") for w in weeks_sorted]
    ax.plot(weeks_sorted, f1_scores, marker="o", markersize=4, label=cls)

ax.plot(
    weeks_sorted,
    [reports[w].get("macro avg", {}).get("f1-score") for w in weeks_sorted],
    color="black",
    linewidth=2,
    linestyle="--",
    label="macro avg",
)

ax.set_xlabel("Week")
ax.set_ylabel("F1 Score")
ax.set_title("Per-class F1 Score over time (full dataset, post-training weeks)")
ax.legend(loc="lower left")
plt.xticks(rotation=90)
plt.tight_layout()
fig

In [ ]:
%matplotlib inline
classes = ["macro avg", "weighted avg"]
months = list(reports.keys())

fig, ax = plt.subplots(figsize=(10, 5))
for cls in classes:
    f1_scores = []
    for month in months:
        report = reports[month]
        f1 = report.get(cls, {}).get("f1-score", None)
        f1_scores.append(f1)
    ax.plot(months, f1_scores, marker="o", label=cls)

ax.set_xlabel("Month")
ax.set_ylabel("F1 Score")
ax.set_title("Per-class F1 Score over time (temporal drift)")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
fig

In [ ]:
baselines = load_baselines()
psi_results = monitor_all_features(
    baselines, df.filter(pl.col("timestamp") > datetime(2025, 9, 1)), FEATURES, period_col="timestamp", class_col=None
)

# Load the permutation importance to weight the features in the drift check
importance_df = load_permutation_importance()
weights = importance_df["drift_weight"].to_dict()

psi_results = add_weighted_psi(psi_results, weights)  # done once

per_feature_flagged = check_drift(psi_results, threshold=0.1, class_col=None)
period_scores = weighted_drift_score(psi_results, period_col="period")

In [ ]:
period_scores

In [ ]:
HIGH_RAW_THRESHOLD = 0.25
LOW_WEIGHTED_THRESHOLD = 0.15

feature_rows = psi_results.filter(pl.col("feature") != "__predicted_class_balance__")

alarm_counts = (
    feature_rows.group_by("period")
    .agg(
        (pl.col("psi") > HIGH_RAW_THRESHOLD).sum().alias("n_flagged_raw_high"),
        (pl.col("weighted_psi") > LOW_WEIGHTED_THRESHOLD).sum().alias("n_flagged_weighted_low"),
    )
    .sort("period")
    .with_columns(pl.col("period").dt.strftime("%Y-%m-%d").alias("week"))
)

alarm_counts

In [ ]:
weeks = alarm_counts["week"].to_list()
raw_alarm_counts = alarm_counts["n_flagged_raw_high"].to_list()
weighted_alarm_counts = alarm_counts["n_flagged_weighted_low"].to_list()

f1_by_week = [reports.get(w, {}).get("macro avg", {}).get("f1-score") for w in weeks]

fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.plot(weeks, f1_by_week, marker="o", color="black", label="Macro F1")
ax1.set_ylabel("F1 Score")
ax1.set_xlabel("Week")
plt.xticks(rotation=90)

ax2 = ax1.twinx()
ax2.plot(
    weeks, raw_alarm_counts, marker="s", color="firebrick", linestyle="--", label="Flagged (raw PSI, high threshold)"
)
ax2.plot(
    weeks,
    weighted_alarm_counts,
    marker="^",
    color="steelblue",
    linestyle="--",
    label="Flagged (weighted PSI, low threshold)",
)
ax2.set_ylabel("Number of features flagged")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.title("Alarm strategy comparison vs. model performance (weekly)")
plt.tight_layout()
fig